# hs-classifier — Full Workflow

This notebook walks through the complete pipeline:

1. **Setup** — configure, build index, load classifier, smoke test
2. **Create eval sample** — take a representative slice of your data
3. **Label** — add ground truth HS codes (simulated here)
4. **Classify + evaluate** — classify the labeled sample and compute metrics
5. **Tune** — compare different model/parameter configurations
6. **Classify full dataset** — run the best config on all your data

## 1. Setup

In [ ]:
import logging
import os
import warnings

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")
for name in ("httpx", "faiss.loader", "sentence_transformers", "numba"):
    logging.getLogger(name).setLevel(logging.WARNING)

import pandas as pd
from sentence_transformers import SentenceTransformer

from hs_classifier import classify_row, init_classifier, init_index

In [ ]:
# one-time: build the FAISS index from Atlas DB (skips if already exists)
init_index()

# load the classifier (FAISS index + S-BERT model)
classifier = init_classifier()

## 2. Create an eval sample

Before classifying your full dataset, take a small representative sample using semantic clustering. The splitter only needs the text column — everything else passes through.

In [ ]:
from hs_classifier.splitter import prepare_eval_sample

df = pd.read_csv("data/raw/ecuador_sample.csv")
print(f"Full dataset: {len(df)} rows")

embed_model = SentenceTransformer(
    os.environ["EMBEDDING_MODEL"], local_files_only=True
)
sample = prepare_eval_sample(
    df, text_col="product_description",
    model=embed_model, sample_frac=0.05,
)

print(f"\nEval sample: {len(sample)} rows ({len(sample)/len(df):.1%})")
sample.groupby("cluster").size().rename("n").sort_index().to_frame()

In [ ]:
# save the sample under the configured intermediate directory
sample.to_csv("data/intermediate/samples/ecuador_sample_eval_5pct.csv", index=False)
print("Saved to: data/intermediate/samples/ecuador_sample_eval_5pct.csv")

## 3. Label the sample

In practice, you'd now open the sample CSV and add an `hs_code` column with the correct HS4 code for each row — either manually, from existing labels, or with a labeling tool.

For this demo, we simulate labeled data with known ground truth:

In [ ]:
# simulated labeled data — in practice this comes from your labeling step
labeled = pd.DataFrame({
    "product_description": [
        "frozen shrimp raw peeled",
        "fresh cut roses red",
        "fresh bananas green cavendish",
        "cacao beans raw fermented",
        "teak wood planks sawn",
        "gold unwrought bullion bars",
        "frozen lobster tails",
        "fresh carnations white",
        "organic bananas fair trade",
        "cocoa beans dried whole",
    ],
    "hs_code": ["0306", "0603", "0803", "1801", "4407", "7108", "0306", "0603", "0803", "1801"],
})
print(f"Labeled sample: {len(labeled)} rows")
labeled

## 4. Classify the labeled sample and evaluate

Run the classifier on each labeled row, collect predictions alongside ground truth, then compute metrics.

In [ ]:
# classify each row and collect predictions
results = []
for row in labeled.to_dict(orient="records"):
    result = classify_row(row, classifier)
    results.append({
        "product_description": row["product_description"],
        "code_true": row["hs_code"],
        **{f"code_{i+1}": c for i, c in enumerate(result.codes)},
        "reason": result.reason,
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
from hs_classifier.evaluator import evaluation_report

report = evaluation_report(results_df, truth_col="code_true")

print(report["summary_text"])
print(report["top1_count"], report["topk_count"], report["chapter_count"])
print("\nCorrectness summary:")
print(report["correctness_summary"])

## 5. Tune and compare

Not happy with the results? Override model and retrieval parameters per call to compare configurations. No need to edit `.env` — just pass arguments to `classify_row()`.

In [ ]:
# compare two reranker models on the same labeled sample
configs = {
    "haiku": {"reranker_model": "anthropic/claude-haiku-4-5-20251001"},
    "gemini": {"reranker_model": "google/gemini-2.5-flash-lite"},
}

for name, params in configs.items():
    run_results = []
    for row in labeled.to_dict(orient="records"):
        result = classify_row(row, classifier, **params)
        run_results.append({
            "code_true": row["hs_code"],
            **{f"code_{i+1}": c for i, c in enumerate(result.codes)},
        })

    run_df = pd.DataFrame(run_results)
    report = evaluation_report(run_df, truth_col="code_true")
    top1 = report["top1_accuracy"]
    topk = report["topk_accuracy"]
    chap = report["chapter_accuracy"]
    print(f"[{name}] Top-1: {top1:.1%}  Top-K: {topk:.1%}  Chapter: {chap:.1%}")

In [ ]:
# or tune retrieval parameters
for top_k in [25, 50, 75]:
    run_results = []
    for row in labeled.to_dict(orient="records"):
        result = classify_row(row, classifier, top_k_total=top_k)
        run_results.append({
            "code_true": row["hs_code"],
            **{f"code_{i+1}": c for i, c in enumerate(result.codes)},
        })

    run_df = pd.DataFrame(run_results)
    report = evaluation_report(run_df, truth_col="code_true")
    top1 = report["top1_accuracy"]
    topk = report["topk_accuracy"]
    print(f"[top_k={top_k}] Top-1: {top1:.1%}  Top-K: {topk:.1%}")

## 6. Classify your full dataset

Once you've picked the best configuration, run it on your full dataset.

In [ ]:
# example: pick the best config from step 5, then classify everything
best_config = {"reranker_model": "google/gemini-2.5-flash-lite"}

df = pd.read_csv("data/raw/ecuador_sample.csv")

all_results = []
for row in df.to_dict(orient="records"):
    result = classify_row(row, classifier, **best_config)
    all_results.append({
        **row,
        **{f"hs_{i+1}": c for i, c in enumerate(result.codes)},
        "reason": result.reason,
    })

classified = pd.DataFrame(all_results)
output_path = "data/raw/ecuador_sample_classified.csv"
classified.to_csv(output_path, index=False)
print(f"Classified {len(classified)} rows")
print(f"Saved to: {output_path}")
classified.head()